# A3-03 · Descriptive holdout of the frozen Exp 0-6 units (release pin)

Release part: Exp 0-6 at the **release pin** (`FINAL_v1/CODE_COMMIT.txt`, `59a9ab7`), `scripts/final.py holdout --exp <exp>`. The pinned code runs in a separate git worktree (`/content/a3_release_<pin>`), in the numerical environment of the
completed units (restored automatically if this Colab differs). Each frozen unit is fitted on the 52 development
participants and evaluated once on the 13 holdout participants; the pinned code refuses any unit that is not in its
frozen record or any other commit, and A3 verifies the master freeze (checksums, pins, QC decisions, store lock, development
results) before and after every call. Resumable. GPU runtime (T4). Descriptive only.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook again
and *Run all* again: finished work is kept on Drive. Everything A3 writes goes to `<outputs>/FINAL_v1/amendments/A3_final/`
(plus, once, the two frozen records created by the freeze); no completed result, pin, QC decision or store is changed.

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the A3 code, installs the requirements. =====
UPDATE_CODE = False   # leave False. True only to move the A3 code pin (<outputs>/FINAL_v1/amendments/A3_final/CODE_COMMIT.txt)
BRANCH = 'claude/a3-final-analysis'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])   # full history: the pins are ancestors
os.chdir('/content/repo')
_run([sys.executable, 'scripts/a3_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')

from adgraph.colab import sh as _sh

def sh(cmd, cwd='/content/repo'):  # every script runs with the file backend
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('/content/repo/configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
A3 = FINAL + '/amendments/A3_final'
print('A3 outputs on Drive:', A3)

## 1. Numerical environment (exactly that of the completed units)

In [ ]:
ACCEPT_ENVIRONMENT_DIFFERENCE = False   # leave False: a difference that cannot be removed must stop here
from adgraph import numerical_env
from adgraph.a3 import environment, inventory as a3_inventory
# The holdout fits are restored to the single environment of the Exp 8a (A2) results (A2 chose it to equal the Exp 7
# units'). Release units developed in a documented-compatible environment (Python patch version, torch cpu / CUDA build)
# are refit in it; `a3.py holdout` compares it with every unit's development environment, records the comparison and
# refuses an incompatible one.
FOUND, _ = a3_inventory.discover(OUT, settings=('xu_dev_cv',))
REFERENCE = environment.holdout_reference(FOUND)
PY = numerical_env.ensure_interpreter(REFERENCE, repo='/content/repo', base='/content',
                                      accept_difference=ACCEPT_ENVIRONMENT_DIFFERENCE, run=sh)
NOW = numerical_env.interpreter_versions(PY, REFERENCE)
DIFF = numerical_env.differences(REFERENCE, NOW)
assert ACCEPT_ENVIRONMENT_DIFFERENCE or not DIFF, f'numerical environment differs from the Exp 8a (A2) environment: {DIFF}'
print('holdout scripts run with', PY, NOW)

## 2. The freeze must verify (refused otherwise)

In [ ]:
sh("python scripts/a3.py verify")

## 3. Holdout of the frozen units of exp0, exp1, exp2, exp3, exp4, exp5, exp6

In [ ]:
ERRORS = []
for exp in ["exp0", "exp1", "exp2", "exp3", "exp4", "exp5", "exp6"]:
    try:
        sh(f"python scripts/a3.py holdout --part release --exp {exp} --python {PY} --repo /content/repo --base /content")
    except RuntimeError:
        ERRORS.append(exp)
sh("python scripts/a3.py holdout-status")
if ERRORS:
    raise RuntimeError(f'holdout units did not complete for {ERRORS}; completed units are saved. Run all again.')